In [1]:
# ============================================================
# NB07_Data_Quality_Checks
# Purpose:
#   Run automated quality tests across Bronze/Silver tables.
#   Write per-check results to gold_data_quality_results
#   and a summary to gold_run_audit.
#   Overall status: HEALTHY / DEGRADED / BLOCKED
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime
import uuid

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
TODAY = datetime.utcnow().date()

print(f"RUN_ID: {RUN_ID}")

checks = []

def add_check(name, category, status, failed_rows, total_rows, details):
    checks.append({
        "check_name": name,
        "check_category": category,
        "status": status,
        "failed_rows": int(failed_rows),
        "total_rows": int(total_rows),
        "details": details,
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP
    })

def tiered(failed, total, warn_pct, fail_pct):
    """Return status from a failed/total ratio."""
    if total == 0:
        return "PASS"
    pct = failed / total
    if pct >= fail_pct:
        return "FAIL"
    if pct >= warn_pct:
        return "WARNING"
    return "PASS"


# ------------------------------------------------------------
# 1. Load tables
# ------------------------------------------------------------

price = spark.read.table("bronze_price_ohlcv")
fund_long = spark.read.table("bronze_fundamentals_long")
fund = spark.read.table("silver_fundamental_features")
regime = spark.read.table("silver_market_regime")

universe = spark.read.table("bronze_universe_snapshot").filter(
    (F.col("instrument_type") == "EQUITY") &
    (F.col("eligible_for_stock_screen") == True)
).select("symbol")

sym_total = universe.count()
print(f"Eligible universe: {sym_total} stocks")




StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 3, Finished, Available, Finished, False)

RUN_ID: ba3f7762-2a92-48e9-83c5-658cf85ac1cf
Eligible universe: 398 stocks


In [2]:
# ------------------------------------------------------------
# 2. PRICE CHECKS
# ------------------------------------------------------------

# --- Check 1: Duplicate keys ---
total_price = price.count()
dup_rows = (
    price.groupBy("symbol", "trade_date").count()
    .filter(F.col("count") > 1)
    .count()
)
add_check(
    "price_duplicate_keys", "PRICE",
    "WARNING" if dup_rows > 0 else "PASS",
    dup_rows, total_price,
    "Duplicate symbol+trade_date in bronze. Silver dedupes on read; run maintenance if count grows."
)

# --- Check 2: OHLC validity ---
invalid_ohlc = price.filter(
    F.col("high").isNull() | F.col("low").isNull() |
    (F.col("high") < F.col("low")) |
    (F.col("high") < F.col("open")) |
    (F.col("high") < F.col("close")) |
    (F.col("low") > F.col("open")) |
    (F.col("low") > F.col("close"))
).count()
add_check(
    "price_ohlc_validity", "PRICE",
    "FAIL" if invalid_ohlc > 0 else "PASS",
    invalid_ohlc, total_price,
    "Rows where High/Low violate OHLC relationships indicate corrupted source data."
)

# --- Check 3: Negative volume (exclude index) ---
neg_volume = price.filter(
    (F.col("symbol") != "NIFTY50") &
    (F.col("volume") < 0)
).count()
add_check(
    "price_negative_volume", "PRICE",
    "FAIL" if neg_volume > 0 else "PASS",
    neg_volume, total_price,
    "Negative volume is impossible and indicates bad rows."
)

# --- Check 4: Zero-volume stock days ---
zero_volume = price.filter(
    (F.col("symbol") != "NIFTY50") &
    (F.col("volume") == 0)
).count()
add_check(
    "price_zero_volume_days", "PRICE",
    tiered(zero_volume, total_price, 0.01, 0.05),
    zero_volume, total_price,
    "Zero-volume days happen (no-trade circuits); large counts suggest ingestion problems."
)

# --- Check 5: Price freshness ---
latest_row = price.agg(F.max("trade_date").alias("max_date")).collect()[0]
latest_date = latest_row["max_date"]
if latest_date is None:
    days_old, fresh_status = 9999, "FAIL"
else:
    days_old = (TODAY - latest_date).days
    fresh_status = "PASS" if days_old <= 4 else ("WARNING" if days_old <= 7 else "FAIL")
add_check(
    "price_freshness", "PRICE", fresh_status,
    0 if fresh_status == "PASS" else days_old, total_price,
    f"Latest bronze trade_date = {latest_date} ({days_old} days old)."
)

# --- Check 6: Price history depth ---
hist = (
    price.filter(F.col("symbol") != "NIFTY50")
    .groupBy("symbol").agg(F.count("*").alias("days"))
)
hist_joined = universe.join(hist, on="symbol", how="left")
short_hist = hist_joined.filter(F.col("days").isNull() | (F.col("days") < 260))
short_count = short_hist.count()
short_examples = ", ".join(
    [r["symbol"] for r in short_hist.limit(5).collect()]
)
add_check(
    "price_history_depth", "PRICE",
    tiered(short_count, sym_total, 0.05, 0.20),
    short_count, sym_total,
    f"{short_count} eligible stocks have <260 price days. Examples: {short_examples or 'none'}"
)




StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 4, Finished, Available, Finished, False)

In [3]:
# ------------------------------------------------------------
# 3. FUNDAMENTALS CHECKS
# ------------------------------------------------------------

# --- Check 7: Coverage — eligible stocks missing fundamentals ---
fund_syms = fund.select("symbol").distinct()
missing_fund = universe.join(fund_syms, on="symbol", how="left_anti")
missing_count = missing_fund.count()
add_check(
    "fundamentals_coverage", "FUNDAMENTALS",
    tiered(missing_count, sym_total, 0.05, 0.20),
    missing_count, sym_total,
    "Eligible stocks with no row in silver_fundamental_features (Yahoo fetch failed or empty)."
)

# --- Check 8: Core field validity ---
fund_elig = fund.join(universe, on="symbol", how="inner")
invalid_core = fund_elig.filter(F.col("fundamentals_valid") == False).count()
add_check(
    "fundamentals_core_fields", "FUNDAMENTALS",
    tiered(invalid_core, sym_total, 0.10, 0.25),
    invalid_core, sym_total,
    "Stocks failing fundamentals_valid (missing EBIT/NI/OCF/CA-CL/Debt/Equity/Mcap)."
)

# --- Check 9: Financial staleness ---
stale = fund_elig.filter(F.col("financial_fresh_flag") == False).count()
add_check(
    "fundamentals_staleness", "FUNDAMENTALS",
    tiered(stale, sym_total, 0.10, 0.30),
    stale, sym_total,
    "Annual financials older than 18 months — excluded from scoring gates anyway."
)

# --- Check 10: NaN / infinite ratio values ---
nan_rows = fund_elig.filter(
    F.isnan("roic") | F.isnan("earnings_yield") |
    F.isnan("pe_proxy") | F.isnan("pb") |
    F.isnan("current_ratio") | F.isnan("debt_to_equity") |
    F.isnan("eps_cagr_3y")
).count()
add_check(
    "fundamentals_nan_values", "FUNDAMENTALS",
    tiered(nan_rows, sym_total, 0.02, 0.10),
    nan_rows, sym_total,
    "NaN ratios (e.g., 0/0) — rows with NaN are excluded from ranks downstream."
)

# --- Check 11: Extreme ratio outliers ---
extreme = fund_elig.filter(
    (F.col("roic").isNotNull() & (F.col("roic") > 2)) |
    (F.col("earnings_yield").isNotNull() & (F.col("earnings_yield") > 1)) |
    (F.col("pe_proxy").isNotNull() & (F.col("pe_proxy") > 300)) |
    (F.col("pb").isNotNull() & (F.col("pb") > 100)) |
    (F.col("current_ratio").isNotNull() & (F.col("current_ratio") > 20))
).count()
add_check(
    "fundamentals_extreme_outliers", "FUNDAMENTALS",
    tiered(extreme, sym_total, 0.02, 0.08),
    extreme, sym_total,
    "ROIC >200%, EY >100%, PE >300, PB >100 or CR >20 — usually data errors, review before trusting ranks."
)

# --- Check 12: Negative enterprise value ---
neg_ev = fund_elig.filter(
    F.col("enterprise_value").isNotNull() &
    (F.col("enterprise_value") <= 0)
).count()
add_check(
    "fundamentals_negative_ev", "FUNDAMENTALS",
    tiered(neg_ev, sym_total, 0.03, 0.10),
    neg_ev, sym_total,
    "Cash exceeds Mcap+Debt — these stocks cannot get an Earnings Yield rank (excluded from Value)."
)

# --- Check 13: Key metric coverage in bronze_fundamentals_long ---
annual_syms = fund_long.filter(F.col("period_type") == "Annual")
key_metrics = [
    "EBIT", "Total Revenue", "Net Income",
    "Operating Cash Flow", "Total Current Assets", "Total Debt"
]
for m in key_metrics:
    cnt = (
        annual_syms
        .filter(F.col("metric_name") == m)
        .select("symbol").distinct().count()
    )
    check_name = "coverage_" + m.lower().replace(" ", "_")
    add_check(
        check_name, "FUNDAMENTALS",
        "PASS" if cnt >= 0.8 * sym_total else
        ("WARNING" if cnt >= 0.5 * sym_total else "FAIL"),
        sym_total - cnt, sym_total,
        f"{cnt}/{sym_total} eligible stocks have annual '{m}' in bronze. "
        "Low coverage usually means a Yahoo field-name mismatch in NB03 METRIC_MAP."
    )




StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 5, Finished, Available, Finished, False)

In [4]:
# ------------------------------------------------------------
# 4. REGIME CHECK
# ------------------------------------------------------------

# --- Check 14: Regime present and fresh ---
reg_count = regime.count()
if reg_count == 0:
    add_check("market_regime_present", "REGIME", "FAIL", 0, 0,
              "silver_market_regime is empty — Nifty 50 ingestion failed in NB02/NB04.")
else:
    latest_reg = regime.orderBy(F.col("trade_date").desc()).first()
    reg_date = latest_reg["trade_date"]
    reg_value = latest_reg["market_regime"]
    reg_days = (TODAY - reg_date).days if reg_date else 9999
    valid_value = reg_value in ("RISK_ON", "RISK_OFF")
    if not valid_value or reg_days > 10:
        reg_status = "FAIL"
    elif reg_days > 7:
        reg_status = "WARNING"
    else:
        reg_status = "PASS"
    add_check(
        "market_regime_present", "REGIME", reg_status,
        0 if reg_status == "PASS" else reg_days, reg_count,
        f"Latest regime = {reg_value} on {reg_date} ({reg_days} days old)."
    )




StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 6, Finished, Available, Finished, False)

In [5]:
# ------------------------------------------------------------
# 5. Write results
# ------------------------------------------------------------

results_df = spark.createDataFrame(checks)

(
    results_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("gold_data_quality_results")
)

# --- Overall audit row ---
status_list = [c["status"] for c in checks]
passed = status_list.count("PASS")
warnings = status_list.count("WARNING")
failed = status_list.count("FAIL")

overall = "BLOCKED" if failed > 0 else ("DEGRADED" if warnings > 0 else "HEALTHY")

audit_df = spark.createDataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "notebook_name": "NB07_Data_Quality_Checks",
    "total_checks": len(checks),
    "passed": passed,
    "warnings": warnings,
    "failed": failed,
    "overall_status": overall
}])

(
       audit_df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable("gold_run_audit")
   )

# --- Console summary ---
print("=" * 55)
print(f"DATA QUALITY SUMMARY — {overall}")
print(f"Checks: {len(checks)} | PASS: {passed} | WARNING: {warnings} | FAIL: {failed}")
print("=" * 55)

if failed > 0 or warnings > 0:
    print("Non-passing checks:")
    for c in checks:
        if c["status"] != "PASS":
            print(f"  [{c['status']}] {c['check_name']}: {c['details']}")

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 7, Finished, Available, Finished, False)

DATA QUALITY SUMMARY — BLOCKED
Checks: 19 | PASS: 17 | WARNING: 0 | FAIL: 2
Non-passing checks:
  [FAIL] price_ohlc_validity: Rows where High/Low violate OHLC relationships indicate corrupted source data.
  [FAIL] coverage_total_current_assets: 0/398 eligible stocks have annual 'Total Current Assets' in bronze. Low coverage usually means a Yahoo field-name mismatch in NB03 METRIC_MAP.


In [6]:
display(
    spark.sql(f"""
        SELECT check_name, check_category, status,
               failed_rows, total_rows, details
        FROM gold_data_quality_results
        WHERE run_id = (
            SELECT run_id FROM gold_run_audit
            ORDER BY run_timestamp DESC LIMIT 1
        )
        ORDER BY
            CASE status WHEN 'FAIL' THEN 1
                        WHEN 'WARNING' THEN 2
                        ELSE 3 END,
            check_category, check_name
    """)
)

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af009e2f-dabb-4cd7-8676-002b235c5cff)

In [7]:
display(
    spark.sql("""
        SELECT run_timestamp, total_checks, passed, warnings, failed, overall_status
        FROM gold_run_audit
        ORDER BY run_timestamp DESC
        LIMIT 10
    """)
)

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12f8b44e-2633-406f-8a24-b703b1d824cd)

In [8]:
display(
    spark.sql("""
        SELECT check_name,
               SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS fail_runs,
               SUM(CASE WHEN status = 'WARNING' THEN 1 ELSE 0 END) AS warn_runs,
               COUNT(*) AS total_runs
        FROM gold_data_quality_results
        GROUP BY check_name
        ORDER BY fail_runs DESC, warn_runs DESC
    """)
)

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e12ebf5-f810-4cee-9e78-c617c4d6c527)

In [9]:
from pyspark.sql import Window
from pyspark.sql import functions as F

price = spark.read.table("bronze_price_ohlcv")
w = Window.partitionBy("symbol", "trade_date").orderBy(F.col("retrieved_at").desc())

(
    price.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1).drop("rn")
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_price_ohlcv")
)
print("Bronze deduplicated (kept latest retrieval per symbol+date).")

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 11, Finished, Available, Finished, False)

Bronze deduplicated (kept latest retrieval per symbol+date).


In [10]:
display(
    spark.sql("""
        SELECT metric_name, COUNT(DISTINCT symbol) AS symbols
        FROM bronze_fundamentals_long
        WHERE period_type = 'Annual'
        GROUP BY metric_name
        ORDER BY symbols DESC
        LIMIT 40
    """)
)

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 62b36a0b-3b9f-4906-bd3d-9be3f626f567)

In [11]:
from pyspark.sql import Window
from pyspark.sql import functions as F

h = spark.read.table("gold_score_history")
w = Window.partitionBy("symbol", "screen_date").orderBy(F.col("run_id").desc())

(
    h.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1).drop("rn")
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_score_history")
)
print("gold_score_history deduplicated (latest run per symbol+screen_date kept).")

StatementMeta(, fd7c9678-2d58-4499-b4a0-0ff492364792, 13, Finished, Available, Finished, False)

gold_score_history deduplicated (latest run per symbol+screen_date kept).
